<a href="https://colab.research.google.com/github/yy3462-create/textual_analysis_project/blob/main/Final_project_NER_to_Network.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# --- CLEAN & PIN (run this first) ---
%pip install -U "pip<24.3" setuptools wheel

# Remove packages that force or prefer NumPy 2.x (not needed for this class)
%pip uninstall -y pytensor opencv-python opencv-contrib-python opencv-python-headless numba cupy-cuda12x tensorflow

# Satisfy IPython's dependency
%pip install "jedi>=0.18.0"

# Pin a NumPy that is ABI-compatible with spaCy wheels
%pip install "numpy==1.26.4"

# Now install the libraries we actually need
%pip install "spacy==3.7.4" "pandas<2.2" "networkx>=3.2" "plotly>=5.18"

print("✅ Clean install complete. NOW go to Runtime → Restart runtime, then run the next cell.")


  Using cached spacy-3.7.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (27 kB)
  Using cached weasel-0.3.4-py3-none-any.whl.metadata (4.7 kB)
Using cached spacy-3.7.4-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (6.5 MB)
Using cached weasel-0.3.4-py3-none-any.whl (50 kB)
  Attempting uninstall: weasel
    Found existing installation: weasel 0.4.3
    Uninstalling weasel-0.4.3:
      Successfully uninstalled weasel-0.4.3
  Attempting uninstall: spacy
    Found existing installation: spacy 3.7.5
    Uninstalling spacy-3.7.5:
      Successfully uninstalled spacy-3.7.5
✅ Clean install complete. NOW go to Runtime → Restart runtime, then run the next cell.


In [ ]:
!pip install -U spacy
!pip install https://github.com/explosion/spacy-models/releases/download/en_core_web_md-3.7.1/en_core_web_md-3.7.1-py3-none-any.whl


# Download a medium English model for better NER
!python -m spacy download en_core_web_md -q

  Using cached spacy-3.8.11-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (27 kB)
  Using cached thinc-8.3.10-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (15 kB)
  Using cached weasel-0.4.3-py3-none-any.whl.metadata (4.6 kB)
  Using cached blis-1.3.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (7.5 kB)
Using cached spacy-3.8.11-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (33.2 MB)
Using cached thinc-8.3.10-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (3.9 MB)
Using cached weasel-0.4.3-py3-none-any.whl (50 kB)
Using cached blis-1.3.3-cp312-cp312-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (11.4 MB)
  Attempting uninstall: blis
    Found existing installation: blis 0.7.11
    Uninstalling blis-0.7.11:
      Successfully uninstalled blis-0.7.11
  Attempting uninstall: weasel
    Found existing installation: weasel 0.3.4
    Uninstalling weasel-0.3.4:
      Successfully uninstalled weasel-

In [ ]:
import spacy
nlp = spacy.load("en_core_web_md")


In [ ]:
# --- Imports (post-restart) ---
import sys, re, numpy as np, pandas as pd, spacy, networkx as nx, plotly.graph_objects as go
from itertools import combinations

# Load spaCy English model
nlp = spacy.load("en_core_web_md")
nlp.max_length = 2_000_000  # or 3_000_000 for extra headroom

# Configure pandas display
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', 50)

# Verify environment
print("✅ Environment ready")
print(f"Python: {sys.version.split()[0]}")
print(f"NumPy: {np.__version__}")
print(f"spaCy: {spacy.__version__}")
print(f"pandas: {pd.__version__}")
print(f"networkx: {nx.__version__}")


✅ Environment ready
Python: 3.12.12
NumPy: 1.26.4
spaCy: 3.7.5
pandas: 2.1.4
networkx: 3.6



## 1) Load your Factiva articles

- Expect a CSV/Parquet with a **`text`** column (one article per row).  
- Optional helpful columns: `article_id`, `date`, `source`, `section`, `headline`.  
- Replace the demo data below with your real path.


In [ ]:
# If using Colab, mount Drive (optional but recommended so outputs persist)
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# 🔁 Replace this with your actual load (e.g., from Drive)
df = pd.read_csv("/content/drive/MyDrive/factiva_ner_project/factiva.csv")

# Demo placeholder (remove once real data is loaded)
# df = pd.DataFrame({
#     "article_id": [1,2,3],
#     "text": [
#         "President Biden met with Ursula von der Leyen in Washington to discuss trade and AI.",
#         "Elon Musk and Tim Cook appeared in Brussels at an EU competition hearing with Margrethe Vestager.",
#         "The IMF and World Bank met in Marrakech; Kristalina Georgieva spoke with Janet Yellen."
#     ],
#     "source": ["ExampleWire","ExampleWire","ExampleWire"],
#     "date": ["2024-09-10","2024-09-12","2024-10-01"]
# })
assert "Headline" in df.columns and df["Headline"].notna().all(), "Data must include a non-null 'text' column."
print(df.head(3))
print(f"Articles loaded: {len(df)}")

   index  \
0      1   
1      1   
2      1   

                                                                                                                                                                                                                                                                                          Headline  \
0                                                                                                                                                           "Other people also shared human experiences with ChatGPT": Young man surprised to notice his podcast's AI coughing as if it were human   
1          Health and Medicine - Oral Health; Findings from Dicle University Has Provided New Information about Oral Health (Comparative performance of large language models in answering periodontology questions from the Turkish Dental Specialty Examination: a cross-sectional study on ...)   
2  Skin Diseases and Conditions - Hyperhidrosis; First Affiliated Hos


## 2) Minimal cleaning + run NER (spaCy)

We keep cleaning light for NER (avoid over-normalizing names). We’ll batch with `nlp.pipe(...)` for speed.


In [ ]:

def clean_text(s: str) -> str:
    return re.sub(r"\s+", " ", str(s)).strip()

df["text_clean"] = df["CombinedText"].map(clean_text)

def iter_docs(texts, batch_size=32):
    for doc in nlp.pipe(texts, batch_size=batch_size, disable=["lemmatizer","textcat"]):
        yield doc

docs = list(iter_docs(df["text_clean"]))
print("Docs processed:", len(docs))


Docs processed: 91



## 3) Extract entities → tidy tables

Keep `PERSON`, `ORG`, `GPE` for policy mapping; attach `article_id` for traceability.


In [ ]:

KEEP = {"PERSON","ORG","GPE"}

rows = []
# Use article_id if present; else fallback to row index
if "article_id" not in df.columns:
    df["article_id"] = np.arange(1, len(df)+1)

for art_id, doc in zip(df["article_id"], docs):
    for ent in doc.ents:
        if ent.label_ in KEEP:
            rows.append({
                "article_id": art_id,
                "entity_raw": ent.text,
                "label": ent.label_,
                "start": ent.start_char,
                "end": ent.end_char
            })
ents_df = pd.DataFrame(rows)

# Normalize a little (preserve display form too)
ents_df["entity_norm"] = (ents_df["entity_raw"]
                          .str.strip()
                          .str.replace(r"\s+", " ", regex=True)
                          .str.replace(r"[’'`]", "'", regex=True))
ents_df["entity_key"] = ents_df["entity_norm"].str.lower()

# Canonical display casing (most frequent)
canonical = (ents_df.groupby("entity_key")["entity_norm"]
             .agg(lambda x: x.value_counts().idxmax())
             .rename("entity"))
ents_df = ents_df.merge(canonical, on="entity_key", how="left")

print("Entities extracted:", len(ents_df))
ents_df.head(10)


Entities extracted: 4929


,article_id,entity_raw,label,start,end,entity_norm,entity_key,entity
0,1,UNITED STATES-,PERSON,0,14,UNITED STATES-,united states-,UNITED STATES-
1,1,AI,ORG,437,439,AI,ai,AI
2,1,AI,ORG,846,848,AI,ai,AI
3,1,AI,ORG,936,938,AI,ai,AI
4,1,Instagram Comments,ORG,1353,1371,Instagram Comments,instagram comments,Instagram Comments
5,2,DEC 5,ORG,5,10,DEC 5,dec 5,DEC 5
6,2,Health & Medicine Week,ORG,63,85,Health & Medicine Week,health & medicine week,Health & Medicine Week
7,2,Health and Medicine - Oral Health,ORG,125,158,Health and Medicine - Oral Health,health and medicine - oral health,Health and Medicine - Oral Health
8,2,Diyarbakir,GPE,203,213,Diyarbakir,diyarbakir,Diyarbakir
9,2,Turkey,GPE,215,221,Turkey,turkey,Turkey



## 4) Quick QA / sanity checks

Look at frequent entities per type. Expect some noise (titles, partial names, acronyms).


In [ ]:

def top_vals(df_, label, n=15):
    s = (df_[df_["label"]==label]["entity"]
         .value_counts()
         .head(n))
    print(f"\nTop {label} entities:")
    display(s)

for lab in ["PERSON","ORG","GPE"]:
    top_vals(ents_df, lab, n=10)



Top PERSON entities:


,count
entity,
MACCALLUM,43
Trump,22
Martha,21
ChatGPT-4o,20
ChatGPT,19
Sam Altman,17
Elon Musk,14
Torres,14
Huang,13



Top ORG entities:


,count
entity,
AI,259
Fund,172
Google,69
ChatGPT,64
OpenAI,58
Meta,46
Nvidia,44
Registrant,44
LLC,42



Top GPE entities:


,count
entity,
U.S.,65
China,27
Mexico,25
US,19
the United States,18
California,13
New York,11
India,11
Korea,11



## 5) Build a PERSON–PERSON co‑mention network

Two people are connected if they appear **in the same article**. (Extension: connect within the same sentence for tighter links.)


In [ ]:

from itertools import combinations

persons = ents_df[ents_df["label"]=="PERSON"][["article_id","entity"]].drop_duplicates()

edge_rows = []
for art_id, group in persons.groupby("article_id"):
    people = sorted(group["entity"].unique())
    for a,b in combinations(people, 2):
        edge_rows.append((a,b,art_id))

edges_df = pd.DataFrame(edge_rows, columns=["src","dst","article_id"])
edge_weights = (edges_df.groupby(["src","dst"]).size()
                .reset_index(name="weight")
                .sort_values("weight", ascending=False))

print(edge_weights.head())
print(f"Edges (unique pairs): {len(edge_weights)} | Articles contributing: {edges_df['article_id'].nunique()}")

# Build graph
G = nx.Graph()
for p in persons["entity"].unique():
    G.add_node(p, type="PERSON")
for _, row in edge_weights.iterrows():
    G.add_edge(row["src"], row["dst"], weight=int(row["weight"]))

print(f"Graph -> Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")


                src            dst  weight
3232      Elon Musk     Sam Altman       7
5857     Sam Altman    Sarah Friar       5
1565    Bret Taylor    Sarah Friar       4
3766  Greg Brockman    Sarah Friar       4
3224      Elon Musk  Paul Nakasone       4
Edges (unique pairs): 6079 | Articles contributing: 81
Graph -> Nodes: 590, Edges: 6079



## 6) Centrality: who matters? who bridges?

Degree centrality (connectivity), betweenness (bridging), and weighted degree (total co‑mentions).


In [ ]:

deg = nx.degree_centrality(G)
btw = nx.betweenness_centrality(G, normalized=True, weight="weight")
deg_w = {n: sum(d["weight"] for _,_,d in G.edges(n, data=True)) for n in G.nodes()}

cent_df = (pd.DataFrame({
    "entity": list(G.nodes()),
    "degree_centrality": [deg[n] for n in G.nodes()],
    "betweenness": [btw[n] for n in G.nodes()],
    "weighted_degree": [deg_w[n] for n in G.nodes()],
}).sort_values(["weighted_degree","degree_centrality"], ascending=False)
  .reset_index(drop=True))

cent_df.head(10)


,entity,degree_centrality,betweenness,weighted_degree
0,Sam Altman,0.168081,0.094696,135
1,David,0.224109,0.107162,132
2,Elon Musk,0.130730,0.093348,112
3,Trump,0.164686,0.068083,97
4,Santa,0.134126,0.078282,79
5,MARTHA MACCALLUM,0.117148,0.000000,69
6,Brian Cole Jr.,0.117148,0.000000,69
7,Paul Mauro,0.117148,0.000000,69
8,David Spunt,0.117148,0.000000,69
9,Martha,0.117148,0.000000,69


In [ ]:
import re
import unicodedata
import pandas as pd
import networkx as nx

# --- 1) Start from your entities table ---
# Expect ents_df with columns: ["article_id","entity","label", ...]
people = ents_df[ents_df["label"]=="PERSON"].copy()

# --- 2) Normalization helpers ---
HONORIFICS = r"(president|pres\.|gov\.|governor|sen\.|senator|rep\.|representative|mr\.|mrs\.|ms\.|dr\.|mayor)"
HONOR_RE = re.compile(rf"^\s*{HONORIFICS}\s+", re.IGNORECASE)

def strip_accents(s: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFKD", s) if not unicodedata.combining(c))

def normalize_person(name: str) -> str:
    if not isinstance(name, str):
        return ""
    n = strip_accents(name).strip()
    n = HONOR_RE.sub("", n)                      # drop titles
    n = re.sub(r"[“”\"(),]", "", n)              # punctuation noise
    n = re.sub(r"\b[A-Z]\.\b", "", n)            # drop middle initials like "R."
    n = re.sub(r"\s+", " ", n).strip()
    # Title case but keep common particles intact
    n = " ".join(w.capitalize() if w.lower() not in {"von","van","de","del","di","la"} else w.lower()
                 for w in n.split())
    return n

people["name_norm"] = people["entity"].map(normalize_person)

# --- 3) Split into full vs last-only ---
def last_name(s: str) -> str:
    parts = s.split()
    return parts[-1] if parts else ""

def is_last_only(s: str) -> bool:
    return len(s.split()) == 1

people["last"] = people["name_norm"].map(last_name)
people["last_only"] = people["name_norm"].map(is_last_only)

# --- 4) Compute dominant full name per last name (with a dominance threshold) ---
fulls = people[~people["last_only"]].copy()
# count full-name mentions by last name
cand = (fulls
        .assign(full=lambda d: d["name_norm"])
        .groupby(["last","full"])
        .size()
        .reset_index(name="n"))

# per-last totals and dominant candidate
totals = cand.groupby("last")["n"].sum().rename("total")
dom = (cand.sort_values(["last","n"], ascending=[True, False])
           .groupby("last").head(1)  # top full per last
           .merge(totals, on="last"))
dom["share"] = dom["n"] / dom["total"]

# choose a threshold; 0.6 = “dominant enough”
DOMINANCE_THRESHOLD = 0.6
dom_map = (dom[dom["share"] >= DOMINANCE_THRESHOLD]
           .set_index("last")["full"]
           .to_dict())

# --- 5) Map last-only mentions to dominant full (when unambiguous) ---
def collapse_name(row):
    nm = row["name_norm"]
    if row["last_only"]:
        last = row["last"]
        if last in dom_map:
            return dom_map[last]   # map "Trump" -> "Donald Trump"
        else:
            return nm              # ambiguous last (e.g., "Bush") → keep as-is
    else:
        return nm                  # already full name

people["name_clean"] = people.apply(collapse_name, axis=1)

# (Optional) See what changed
# people.loc[people["name_norm"]!=people["name_clean"], ["name_norm","name_clean"]].drop_duplicates().head(20)


In [ ]:
# Rebuild PERSON table with cleaned names
persons_clean = (people[["article_id","name_clean"]]
                 .rename(columns={"name_clean":"entity"})
                 .drop_duplicates())

# Recompute co-mentions with cleaned entities
from itertools import combinations
edge_rows = []
for art_id, grp in persons_clean.groupby("article_id"):
    ppl = sorted(grp["entity"].unique())
    for a, b in combinations(ppl, 2):
        edge_rows.append((a, b, art_id))

edges_df = pd.DataFrame(edge_rows, columns=["src","dst","article_id"])
edge_weights = (edges_df.groupby(["src","dst"]).size()
                .reset_index(name="weight")
                .sort_values("weight", ascending=False))

# Build graph and recalc centralities
G = nx.Graph()
for p in persons_clean["entity"].unique():
    G.add_node(p, type="PERSON")
for _, r in edge_weights.iterrows():
    G.add_edge(r["src"], r["dst"], weight=int(r["weight"]))

deg = nx.degree_centrality(G)
btw = nx.betweenness_centrality(G, normalized=True, weight="weight")
deg_w = {n: sum(d["weight"] for _,_,d in G.edges(n, data=True)) for n in G.nodes()}

cent_df = (pd.DataFrame({
    "entity": list(G.nodes()),
    "degree_centrality": [deg[n] for n in G.nodes()],
    "betweenness": [btw[n] for n in G.nodes()],
    "weighted_degree": [deg_w[n] for n in G.nodes()],
}).sort_values(["weighted_degree","degree_centrality"], ascending=False)
  .reset_index(drop=True))

cent_df.head(10)


,entity,degree_centrality,betweenness,weighted_degree
0,Sam Altman,0.161410,0.082378,121
1,David,0.218924,0.101326,118
2,Elon Musk,0.128015,0.117176,100
3,Donald Trump,0.163265,0.097275,88
4,Santa,0.131725,0.070715,71
5,Martha Maccallum,0.113173,0.000000,61
6,Brian Cole Jr.,0.113173,0.000000,61
7,Paul Mauro,0.113173,0.000000,61
8,David Spunt,0.113173,0.000000,61
9,Martha,0.113173,0.000000,61



## 7) Interactive network (Plotly)

Small/medium corpora render fine inline. For larger projects, export to **Gephi**.


In [ ]:
# === Show only the TOP-K strongest relationships (by edge "weight") ===
TOP_K = 20  # change as needed

# 1) Pick top-K edges by weight
edges_sorted = sorted(
    G.edges(data=True),
    key=lambda e: e[2].get("weight", 1),
    reverse=True
)
top_edges = edges_sorted[:TOP_K]

# 2) Build a subgraph with just those edges (and their incident nodes)
H = nx.Graph()
H.add_nodes_from(G.nodes(data=True))  # keep node attrs if any
for u, v, d in top_edges:
    H.add_edge(u, v, **d)

# Optional: remove isolated nodes (if any) that snuck in without edges
H.remove_nodes_from(list(nx.isolates(H)))

# 3) Recompute layout and centralities on the subgraph
pos = nx.spring_layout(H, k=0.6, seed=42, weight="weight")

deg_cen_map = nx.degree_centrality(H)
btw_map = nx.betweenness_centrality(H, normalized=True, weight="weight")
wdeg_map = {n: sum(d["weight"] for _,_,d in H.edges(n, data=True)) for n in H.nodes()}

# 4) Build Plotly traces (edges first)
edge_x, edge_y = [], []
for u, v, d in H.edges(data=True):
    x0, y0 = pos[u]; x1, y1 = pos[v]
    edge_x += [x0, x1, None]
    edge_y += [y0, y1, None]

edge_trace = go.Scatter(
    x=edge_x, y=edge_y, mode='lines',
    line=dict(width=0.5),
    hoverinfo='none'
)

# 5) Nodes
node_x = [pos[n][0] for n in H.nodes()]
node_y = [pos[n][1] for n in H.nodes()]
node_sizes = [8 + 12*deg_cen_map.get(n, 0) for n in H.nodes()]
node_text = [
    f"{n}<br>degree={deg_cen_map.get(n,0):.3f}"
    f"<br>betweenness={btw_map.get(n,0):.3f}"
    f"<br>w_degree={wdeg_map.get(n,0)}"
    for n in H.nodes()
]

node_trace = go.Scatter(
    x=node_x, y=node_y, mode='markers+text',
    text=[n for n in H.nodes()],
    textposition="top center",
    marker=dict(size=node_sizes),
    hovertext=node_text, hoverinfo='text'
)

fig = go.Figure(data=[edge_trace, node_trace])
fig.update_layout(
    title=f"Top {min(TOP_K, H.number_of_edges())} Person–Person Relationships (by co-mentions)",
    showlegend=False, height=640,
    margin=dict(l=20, r=20, t=50, b=20),
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False)
)
fig.show()
